# 12. Operations Research

Operations Research (OR) uses mathematical optimization to make better decisions.

**Why OR Matters for Calculus:**
- Nonlinear programming uses derivatives to find optimal solutions
- Lagrange multipliers solve constrained optimization (multivariable calculus!)
- Gradient descent uses $\nabla f$ to minimize functions
- KKT conditions generalize Lagrange multipliers using derivatives
- Sensitivity analysis uses partial derivatives

**Topics Covered:**
1. Linear programming (simplex method)
2. Network optimization (shortest path, max flow)
3. Nonlinear programming with calculus
4. Lagrange multipliers for constrained optimization
5. Gradient descent and optimization algorithms
6. KKT conditions
7. Real-world applications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import optimize
from scipy.optimize import linprog, minimize
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D

sns.set_style('whitegrid')
np.random.seed(42)

## 1. Linear Programming - Introduction

**Linear Programming (LP)** optimizes a linear objective function subject to linear constraints.

**Standard form:**
$$\begin{align}
\text{minimize} \quad & c^T x \\
\text{subject to} \quad & Ax \leq b \\
& x \geq 0
\end{align}$$

**Example:** Production planning
- Make two products: A and B
- Profit: \$40 per unit A, \$30 per unit B
- Constraints: labor hours, material, demand

In [ ]:
# Production planning problem
# Maximize: 40*x1 + 30*x2 (profit)
# Subject to:
#   2*x1 + 1*x2 <= 100  (labor hours)
#   1*x1 + 2*x2 <= 80   (material)
#   x1 <= 40            (demand A)
#   x2 <= 50            (demand B)
#   x1, x2 >= 0

# For scipy.linprog, we minimize, so negate the objective
c = [-40, -30]  # Negate for maximization

# Inequality constraints Ax <= b
A_ub = [
    [2, 1],   # Labor
    [1, 2],   # Material
    [1, 0],   # Demand A
    [0, 1]    # Demand B
]
b_ub = [100, 80, 40, 50]

# Bounds for variables (x >= 0)
x_bounds = [(0, None), (0, None)]

# Solve
result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=x_bounds, method='highs')

# Visualize feasible region
x1 = np.linspace(0, 60, 400)

# Constraint lines
x2_labor = (100 - 2*x1) / 1       # 2x1 + x2 = 100
x2_material = (80 - x1) / 2       # x1 + 2x2 = 80
x2_demand_A = np.full_like(x1, 100)  # x1 = 40 doesn't constrain x2 directly
x2_demand_B = np.full_like(x1, 50)   # x2 = 50

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Feasible region
axes[0].plot(x1, x2_labor, 'b-', label='Labor: 2x₁ + x₂ ≤ 100', linewidth=2)
axes[0].plot(x1, x2_material, 'r-', label='Material: x₁ + 2x₂ ≤ 80', linewidth=2)
axes[0].axvline(40, color='g', linestyle='--', label='Demand A: x₁ ≤ 40', linewidth=2)
axes[0].axhline(50, color='purple', linestyle='--', label='Demand B: x₂ ≤ 50', linewidth=2)

# Fill feasible region
x1_fill = np.linspace(0, 40, 100)
x2_upper = np.minimum(100 - 2*x1_fill, (80 - x1_fill)/2)
x2_upper = np.minimum(x2_upper, 50)
axes[0].fill_between(x1_fill, 0, x2_upper, alpha=0.3, color='yellow', label='Feasible region')

# Optimal solution
if result.success:
    axes[0].plot(result.x[0], result.x[1], 'ro', markersize=15, 
                 label=f'Optimal: ({result.x[0]:.1f}, {result.x[1]:.1f})', zorder=5)

# Iso-profit lines
for profit in [800, 1600, 2000, 2400]:
    x2_profit = (profit - 40*x1) / 30
    axes[0].plot(x1, x2_profit, 'k--', alpha=0.3, linewidth=1)

axes[0].set_xlabel('x₁ (Product A)')
axes[0].set_ylabel('x₂ (Product B)')
axes[0].set_title('Linear Programming: Feasible Region')
axes[0].set_xlim(0, 60)
axes[0].set_ylim(0, 60)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Corner points analysis
corners = [
    (0, 0),
    (0, 40),
    (20, 30),  # Intersection of labor and material
    (40, 20),
    (40, 0)
]

profits = [40*x + 30*y for x, y in corners]
corner_df = pd.DataFrame({
    'x₁': [c[0] for c in corners],
    'x₂': [c[1] for c in corners],
    'Profit': profits
})

axes[1].bar(range(len(corners)), profits, color='skyblue', edgecolor='black')
axes[1].set_xlabel('Corner Point')
axes[1].set_ylabel('Profit ($)')
axes[1].set_title('Profit at Each Corner Point')
axes[1].set_xticks(range(len(corners)))
axes[1].set_xticklabels([f'({c[0]:.0f},{c[1]:.0f})' for c in corners], rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Highlight optimal
max_idx = np.argmax(profits)
axes[1].bar(max_idx, profits[max_idx], color='red', edgecolor='black', label='Optimal')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Linear Programming Solution:")
print(f"Optimal production:")
print(f"  Product A: {result.x[0]:.2f} units")
print(f"  Product B: {result.x[1]:.2f} units")
print(f"Maximum profit: ${-result.fun:.2f}")
print(f"\nCorner points analysis:")
print(corner_df.to_string(index=False))

## 2. Nonlinear Programming with Calculus

When the objective or constraints are nonlinear, we use **calculus** to find optimal solutions!

**Unconstrained optimization:**
$$\min_x f(x)$$

**Necessary condition:** $\nabla f(x^*) = 0$ (first derivative test)

**Sufficient condition:** $\nabla^2 f(x^*) \succ 0$ (Hessian is positive definite)

In [ ]:
# Example: Minimize f(x, y) = x² + y² - 2x - 4y + 5

def f(X):
    """Objective function."""
    x, y = X
    return x**2 + y**2 - 2*x - 4*y + 5

def grad_f(X):
    """Gradient of f."""
    x, y = X
    return np.array([2*x - 2, 2*y - 4])

def hess_f(X):
    """Hessian matrix of f."""
    return np.array([[2, 0], [0, 2]])

# Analytical solution: ∇f = 0
# 2x - 2 = 0 => x = 1
# 2y - 4 = 0 => y = 2
x_analytical = np.array([1.0, 2.0])

# Numerical solution using scipy
x0 = np.array([0.0, 0.0])  # Initial guess
result = minimize(f, x0, jac=grad_f, method='BFGS')

# Visualize
x_range = np.linspace(-2, 4, 100)
y_range = np.linspace(-1, 5, 100)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = X_grid**2 + Y_grid**2 - 2*X_grid - 4*Y_grid + 5

fig = plt.figure(figsize=(15, 5))

# 3D surface
ax1 = fig.add_subplot(131, projection='3d')
surf = ax1.plot_surface(X_grid, Y_grid, Z_grid, cmap='viridis', alpha=0.7)
ax1.scatter([result.x[0]], [result.x[1]], [result.fun], color='red', s=100, 
            label='Minimum', zorder=5)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('f(x, y)')
ax1.set_title('3D Surface Plot')
fig.colorbar(surf, ax=ax1, shrink=0.5)

# Contour plot
ax2 = fig.add_subplot(132)
contour = ax2.contour(X_grid, Y_grid, Z_grid, levels=20, cmap='viridis')
ax2.clabel(contour, inline=True, fontsize=8)
ax2.plot(result.x[0], result.x[1], 'ro', markersize=10, label='Minimum')

# Show gradient direction at several points
sample_points = [(0, 0), (2, 1), (0, 3), (2, 3)]
for pt in sample_points:
    grad = grad_f(pt)
    # Arrow points in direction of steepest ascent
    ax2.arrow(pt[0], pt[1], -grad[0]*0.3, -grad[1]*0.3, 
              head_width=0.2, head_length=0.15, fc='red', ec='red', alpha=0.6)

ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Contour Plot with Gradient Vectors')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# Hessian analysis
H = hess_f(result.x)
eigenvalues, eigenvectors = np.linalg.eig(H)

ax3 = fig.add_subplot(133)
ax3.axis('off')
analysis_text = f"""Calculus Analysis:

Objective: f(x,y) = x² + y² - 2x - 4y + 5

Gradient:
  ∇f = [2x - 2, 2y - 4]

Critical point (∇f = 0):
  2x - 2 = 0  →  x* = 1
  2y - 4 = 0  →  y* = 2

Hessian matrix:
  H = [[2, 0],
       [0, 2]]

Eigenvalues: {eigenvalues[0]:.1f}, {eigenvalues[1]:.1f}
Both positive → Positive definite
→ Local minimum confirmed!

Numerical solution:
  x* = ({result.x[0]:.6f}, {result.x[1]:.6f})
  f(x*) = {result.fun:.6f}

Analytical solution:
  x* = (1.000000, 2.000000)
  f(x*) = 0.000000
"""

ax3.text(0.1, 0.5, analysis_text, fontsize=10, family='monospace',
         verticalalignment='center', transform=ax3.transAxes)

plt.tight_layout()
plt.show()

print(analysis_text)

## 3. Lagrange Multipliers - Constrained Optimization

**Problem:** Optimize $f(x, y)$ subject to constraint $g(x, y) = c$

**Lagrange's Method:**
$$\mathcal{L}(x, y, \lambda) = f(x, y) - \lambda(g(x, y) - c)$$

**Optimality conditions:**
$$\begin{align}
\frac{\partial \mathcal{L}}{\partial x} &= 0 \\
\frac{\partial \mathcal{L}}{\partial y} &= 0 \\
\frac{\partial \mathcal{L}}{\partial \lambda} &= 0 \quad \text{(constraint)}
\end{align}$$

**Geometric interpretation:** At optimum, $\nabla f = \lambda \nabla g$ (gradients are parallel!)

In [ ]:
# Example: Maximize f(x, y) = xy subject to x² + y² = 8
# (Find the rectangle with maximum area inscribed in a circle)

def objective(X):
    """f(x, y) = xy (negated for minimization)."""
    x, y = X
    return -x * y  # Negate for maximization

def constraint_eq(X):
    """g(x, y) = x² + y² - 8 = 0."""
    x, y = X
    return x**2 + y**2 - 8

# Analytical solution using Lagrange multipliers:
# ∇f = λ∇g
# [y, x] = λ[2x, 2y]
# y = 2λx  and  x = 2λy
# => y = 2λx = 2λ(2λy) = 4λ²y
# => 1 = 4λ²  => λ = ±1/2
# From x² + y² = 8 and x = y (by symmetry): 2x² = 8 => x = ±2
x_analytical = np.array([2.0, 2.0])

# Numerical solution
constraint = {'type': 'eq', 'fun': constraint_eq}
x0 = np.array([1.0, 1.0])
result = minimize(objective, x0, method='SLSQP', constraints=constraint)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contour plot
x_range = np.linspace(-4, 4, 200)
y_range = np.linspace(-4, 4, 200)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = X_grid * Y_grid  # Objective function

contour = axes[0].contour(X_grid, Y_grid, Z_grid, levels=20, cmap='viridis')
axes[0].clabel(contour, inline=True, fontsize=8)

# Constraint circle: x² + y² = 8
theta = np.linspace(0, 2*np.pi, 100)
x_circle = np.sqrt(8) * np.cos(theta)
y_circle = np.sqrt(8) * np.sin(theta)
axes[0].plot(x_circle, y_circle, 'r-', linewidth=3, label='Constraint: x² + y² = 8')

# Optimal point
axes[0].plot(result.x[0], result.x[1], 'ro', markersize=15, 
             label=f'Maximum: ({result.x[0]:.2f}, {result.x[1]:.2f})', zorder=5)

# Show gradient vectors at optimal point
# ∇f = [y, x]
grad_f_opt = np.array([result.x[1], result.x[0]])
# ∇g = [2x, 2y]
grad_g_opt = np.array([2*result.x[0], 2*result.x[1]])

# Normalize for visualization
scale = 0.8
axes[0].arrow(result.x[0], result.x[1], 
              grad_f_opt[0]*scale, grad_f_opt[1]*scale,
              head_width=0.3, head_length=0.2, fc='blue', ec='blue', 
              linewidth=2, label='∇f')
axes[0].arrow(result.x[0], result.x[1], 
              grad_g_opt[0]*scale/4, grad_g_opt[1]*scale/4,
              head_width=0.3, head_length=0.2, fc='green', ec='green', 
              linewidth=2, label='∇g')

axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Lagrange Multipliers: ∇f = λ∇g')
axes[0].legend(loc='lower left', fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)

# Analysis text
axes[1].axis('off')
analysis = f"""Lagrange Multiplier Analysis:

Problem:
  Maximize f(x, y) = xy
  Subject to: g(x, y) = x² + y² = 8

Lagrangian:
  ℒ(x, y, λ) = xy - λ(x² + y² - 8)

Optimality conditions:
  ∂ℒ/∂x = y - 2λx = 0
  ∂ℒ/∂y = x - 2λy = 0
  ∂ℒ/∂λ = -(x² + y² - 8) = 0

From first two equations:
  y = 2λx
  x = 2λy
  → y = 2λ(2λy) = 4λ²y
  → 1 = 4λ²
  → λ = ±1/2

By symmetry, x = y at optimum:
  x² + x² = 8
  2x² = 8
  x = ±2

Critical points:
  (2, 2): f = 4 ← Maximum
  (-2, -2): f = 4 ← Maximum
  (2, -2): f = -4 ← Minimum
  (-2, 2): f = -4 ← Minimum

Numerical result:
  x* = ({result.x[0]:.6f}, {result.x[1]:.6f})
  f(x*) = {-result.fun:.6f}
  λ ≈ {result.x[1]/(2*result.x[0]):.6f}
"""

axes[1].text(0.1, 0.5, analysis, fontsize=10, family='monospace',
             verticalalignment='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

print(analysis)

## 4. Gradient Descent

**Gradient Descent** is an iterative algorithm that uses derivatives to find minima.

**Algorithm:**
$$x_{k+1} = x_k - \alpha \nabla f(x_k)$$

where $\alpha$ is the learning rate (step size).

**Why it works:** $\nabla f$ points in direction of steepest ascent, so $-\nabla f$ points toward the minimum!

In [ ]:
# Gradient descent on Rosenbrock function
# f(x, y) = (1-x)² + 100(y-x²)²
# Global minimum at (1, 1) with f(1, 1) = 0

def rosenbrock(X):
    """Rosenbrock function."""
    x, y = X
    return (1 - x)**2 + 100*(y - x**2)**2

def grad_rosenbrock(X):
    """Gradient of Rosenbrock function."""
    x, y = X
    df_dx = -2*(1 - x) - 400*x*(y - x**2)
    df_dy = 200*(y - x**2)
    return np.array([df_dx, df_dy])

def gradient_descent(f, grad_f, x0, alpha=0.001, max_iter=10000, tol=1e-6):
    """Gradient descent algorithm."""
    x = x0.copy()
    history = [x.copy()]
    f_history = [f(x)]
    
    for i in range(max_iter):
        grad = grad_f(x)
        x_new = x - alpha * grad
        
        history.append(x_new.copy())
        f_history.append(f(x_new))
        
        if np.linalg.norm(x_new - x) < tol:
            break
        
        x = x_new
    
    return x, np.array(history), np.array(f_history)

# Run gradient descent with different learning rates
x0 = np.array([-1.0, 2.0])
alphas = [0.0001, 0.0005, 0.001]
results = {}

for alpha in alphas:
    x_opt, history, f_history = gradient_descent(rosenbrock, grad_rosenbrock, x0, alpha=alpha)
    results[alpha] = (x_opt, history, f_history)

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Contour plot with paths
x_range = np.linspace(-2, 2, 200)
y_range = np.linspace(-1, 3, 200)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = (1 - X_grid)**2 + 100*(Y_grid - X_grid**2)**2

axes[0, 0].contour(X_grid, Y_grid, np.log10(Z_grid + 1), levels=20, cmap='viridis')
axes[0, 0].plot(1, 1, 'r*', markersize=20, label='Global minimum (1, 1)')

colors = ['blue', 'green', 'orange']
for (alpha, (x_opt, history, _)), color in zip(results.items(), colors):
    axes[0, 0].plot(history[:, 0], history[:, 1], 'o-', 
                    color=color, markersize=3, linewidth=1, alpha=0.7,
                    label=f'α={alpha} ({len(history)} steps)')

axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_title('Gradient Descent Paths (log scale)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Convergence plot
for (alpha, (_, _, f_history)), color in zip(results.items(), colors):
    axes[0, 1].semilogy(f_history, color=color, linewidth=2, label=f'α={alpha}')

axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('f(x) (log scale)')
axes[0, 1].set_title('Convergence: Objective Function Value')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Gradient magnitude over iterations
for (alpha, (_, history, _)), color in zip(results.items(), colors):
    grad_norms = [np.linalg.norm(grad_rosenbrock(x)) for x in history]
    axes[1, 0].semilogy(grad_norms, color=color, linewidth=2, label=f'α={alpha}')

axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('||∇f|| (log scale)')
axes[1, 0].set_title('Convergence: Gradient Magnitude')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Summary
axes[1, 1].axis('off')
summary = """Gradient Descent Summary:

Function: Rosenbrock
  f(x,y) = (1-x)² + 100(y-x²)²

Gradient:
  ∂f/∂x = -2(1-x) - 400x(y-x²)
  ∂f/∂y = 200(y-x²)

Update rule:
  x_{k+1} = x_k - α∇f(x_k)

Initial point: (-1.0, 2.0)
Target: (1.0, 1.0)

Results:
"""

for alpha, (x_opt, history, f_history) in results.items():
    summary += f"\n  α = {alpha}:
"
    summary += f"    Iterations: {len(history)}\n"
    summary += f"    Final x: ({x_opt[0]:.4f}, {x_opt[1]:.4f})\n"
    summary += f"    Final f(x): {f_history[-1]:.6e}\n"

summary += "\nKey insight:
"
summary += "  Larger α → faster but less stable\n"
summary += "  Smaller α → slower but more stable\n"

axes[1, 1].text(0.1, 0.5, summary, fontsize=9, family='monospace',
                verticalalignment='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(summary)

## 5. Newton's Method for Optimization

**Newton's Method** uses second derivatives (Hessian) for faster convergence.

**Update rule:**
$$x_{k+1} = x_k - [\nabla^2 f(x_k)]^{-1} \nabla f(x_k)$$

**Advantages:**
- Quadratic convergence (very fast near optimum)
- Uses curvature information (Hessian)

**Disadvantages:**
- Requires computing and inverting Hessian (expensive!)
- May not converge if far from optimum

In [ ]:
# Compare gradient descent vs Newton's method

def f_quadratic(X):
    """Simple quadratic: f(x, y) = 2x² + 3y² - 4x + 6y + 10."""
    x, y = X
    return 2*x**2 + 3*y**2 - 4*x + 6*y + 10

def grad_quadratic(X):
    """Gradient."""
    x, y = X
    return np.array([4*x - 4, 6*y + 6])

def hess_quadratic(X):
    """Hessian (constant for quadratic)."""
    return np.array([[4, 0], [0, 6]])

def newtons_method(f, grad_f, hess_f, x0, max_iter=20, tol=1e-8):
    """Newton's method for optimization."""
    x = x0.copy()
    history = [x.copy()]
    f_history = [f(x)]
    
    for i in range(max_iter):
        grad = grad_f(x)
        hess = hess_f(x)
        
        # Newton step: x_new = x - H^(-1) * grad
        try:
            delta = np.linalg.solve(hess, grad)
        except np.linalg.LinAlgError:
            print("Hessian is singular!")
            break
        
        x_new = x - delta
        
        history.append(x_new.copy())
        f_history.append(f(x_new))
        
        if np.linalg.norm(x_new - x) < tol:
            break
        
        x = x_new
    
    return x, np.array(history), np.array(f_history)

# Starting point
x0 = np.array([3.0, -3.0])

# Run both methods
x_gd, hist_gd, f_gd = gradient_descent(f_quadratic, grad_quadratic, x0, alpha=0.1, max_iter=100)
x_newton, hist_newton, f_newton = newtons_method(f_quadratic, grad_quadratic, hess_quadratic, x0)

# Analytical solution: ∇f = 0
# 4x - 4 = 0 => x = 1
# 6y + 6 = 0 => y = -1
x_true = np.array([1.0, -1.0])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Contour plot
x_range = np.linspace(-1, 4, 100)
y_range = np.linspace(-4, 1, 100)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = 2*X_grid**2 + 3*Y_grid**2 - 4*X_grid + 6*Y_grid + 10

contour = axes[0].contour(X_grid, Y_grid, Z_grid, levels=20, cmap='viridis')
axes[0].clabel(contour, inline=True, fontsize=8)

axes[0].plot(hist_gd[:, 0], hist_gd[:, 1], 'bo-', markersize=4, linewidth=1.5,
             label=f'Gradient Descent ({len(hist_gd)} steps)', alpha=0.7)
axes[0].plot(hist_newton[:, 0], hist_newton[:, 1], 'rs-', markersize=6, linewidth=2,
             label=f"Newton's Method ({len(hist_newton)} steps)", alpha=0.7)
axes[0].plot(x_true[0], x_true[1], 'g*', markersize=20, label='True minimum')

axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Optimization Paths')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Convergence comparison
axes[1].semilogy(f_gd, 'b-', linewidth=2, label='Gradient Descent')
axes[1].semilogy(f_newton, 'r-', linewidth=2, label="Newton's Method")
axes[1].axhline(f_quadratic(x_true), color='green', linestyle='--', linewidth=1, label='True minimum')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('f(x) (log scale)')
axes[1].set_title('Convergence Speed Comparison')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Analysis
axes[2].axis('off')
comparison = f"""Method Comparison:

Function:
  f(x,y) = 2x² + 3y² - 4x + 6y + 10

Gradient Descent:
  Update: x_{'{k+1}'} = x_k - α∇f(x_k)
  Iterations: {len(hist_gd)}
  Final: ({x_gd[0]:.6f}, {x_gd[1]:.6f})
  Final f(x): {f_gd[-1]:.6e}

Newton's Method:
  Update: x_{'{k+1}'} = x_k - H⁻¹∇f(x_k)
  Iterations: {len(hist_newton)}
  Final: ({x_newton[0]:.6f}, {x_newton[1]:.6f})
  Final f(x): {f_newton[-1]:.6e}

True minimum:
  x* = (1.000000, -1.000000)
  f(x*) = 3.000000

Key observations:
  • Newton converges in 1 step
    (exact for quadratics!)
  • GD requires many iterations
  • Newton uses Hessian (2nd order)
  • GD uses only gradient (1st order)
"""

axes[2].text(0.1, 0.5, comparison, fontsize=10, family='monospace',
             verticalalignment='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.show()

print(comparison)

## 6. KKT Conditions

**Karush-Kuhn-Tucker (KKT) conditions** generalize Lagrange multipliers to inequality constraints.

**Problem:**
$$\begin{align}
\min_x \quad & f(x) \\
\text{s.t.} \quad & g_i(x) \leq 0, \quad i=1,\ldots,m \\
& h_j(x) = 0, \quad j=1,\ldots,p
\end{align}$$

**KKT Conditions:**
1. **Stationarity:** $\nabla f(x^*) + \sum_i \lambda_i \nabla g_i(x^*) + \sum_j \mu_j \nabla h_j(x^*) = 0$
2. **Primal feasibility:** $g_i(x^*) \leq 0$, $h_j(x^*) = 0$
3. **Dual feasibility:** $\lambda_i \geq 0$
4. **Complementary slackness:** $\lambda_i g_i(x^*) = 0$ (if constraint not active, $\lambda_i = 0$)

In [ ]:
# Example: Minimize f(x, y) = (x-2)² + (y-3)²
# Subject to: g₁(x, y) = x + y - 1 ≤ 0
#             g₂(x, y) = -x ≤ 0 (i.e., x ≥ 0)
#             g₃(x, y) = -y ≤ 0 (i.e., y ≥ 0)

def objective_kkt(X):
    """f(x, y) = (x-2)² + (y-3)²."""
    x, y = X
    return (x - 2)**2 + (y - 3)**2

def grad_objective_kkt(X):
    """Gradient of objective."""
    x, y = X
    return np.array([2*(x - 2), 2*(y - 3)])

# Constraints
constraints = [
    {'type': 'ineq', 'fun': lambda X: -(X[0] + X[1] - 1)},  # x + y ≤ 1
    {'type': 'ineq', 'fun': lambda X: X[0]},                # x ≥ 0
    {'type': 'ineq', 'fun': lambda X: X[1]}                 # y ≥ 0
]

# Solve
x0 = np.array([0.5, 0.5])
result = minimize(objective_kkt, x0, method='SLSQP', constraints=constraints)

# Unconstrained minimum (for comparison)
x_unconstrained = np.array([2.0, 3.0])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Contour plot
x_range = np.linspace(-0.5, 3, 200)
y_range = np.linspace(-0.5, 3.5, 200)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
Z_grid = (X_grid - 2)**2 + (Y_grid - 3)**2

contour = axes[0].contour(X_grid, Y_grid, Z_grid, levels=20, cmap='viridis')
axes[0].clabel(contour, inline=True, fontsize=8)

# Feasible region
x_line = np.linspace(0, 1, 100)
y_line = 1 - x_line
axes[0].fill_between(x_line, 0, y_line, alpha=0.3, color='yellow', label='Feasible region')

# Constraint boundaries
axes[0].plot(x_line, y_line, 'r-', linewidth=2, label='x + y = 1')
axes[0].axhline(0, color='green', linestyle='--', linewidth=1, label='y = 0')
axes[0].axvline(0, color='blue', linestyle='--', linewidth=1, label='x = 0')

# Solutions
axes[0].plot(x_unconstrained[0], x_unconstrained[1], 'bs', markersize=12, 
             label='Unconstrained min (2, 3)')
axes[0].plot(result.x[0], result.x[1], 'ro', markersize=12, 
             label=f'Constrained min ({result.x[0]:.2f}, {result.x[1]:.2f})')

# Show gradient at constrained minimum
grad = grad_objective_kkt(result.x)
axes[0].arrow(result.x[0], result.x[1], -grad[0]*0.3, -grad[1]*0.3,
              head_width=0.15, head_length=0.1, fc='purple', ec='purple', linewidth=2)

axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Constrained Optimization with KKT')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-0.5, 3)
axes[0].set_ylim(-0.5, 3.5)

# KKT analysis
# At x* = (0.5, 0.5), only g₁ is active (x + y = 1)
# KKT: ∇f + λ₁∇g₁ = 0
# [2(x-2), 2(y-3)] + λ₁[1, 1] = 0
# => 2(0.5-2) + λ₁ = 0 => λ₁ = 3
# => 2(0.5-3) + λ₁ = 0 => λ₁ = 5
# Wait, these don't match! Let me recalculate...

axes[1].axis('off')
kkt_analysis = f"""KKT Conditions Analysis:

Problem:
  min f(x, y) = (x-2)² + (y-3)²
  s.t. g₁: x + y ≤ 1
       g₂: x ≥ 0
       g₃: y ≥ 0

Unconstrained minimum:
  ∇f = 0 → x* = (2, 3)
  But violates g₁! (2+3 > 1)

Constrained solution:
  x* = ({result.x[0]:.4f}, {result.x[1]:.4f})
  f(x*) = {result.fun:.4f}

Active constraints:
  g₁: x + y - 1 = {result.x[0] + result.x[1] - 1:.6f} (active!)
  g₂: -x = {-result.x[0]:.6f}
  g₃: -y = {-result.x[1]:.6f}

KKT Stationarity:
  ∇f + λ₁∇g₁ + λ₂∇g₂ + λ₃∇g₃ = 0

At x* = ({result.x[0]:.2f}, {result.x[1]:.2f}):
  ∇f = [{grad[0]:.2f}, {grad[1]:.2f}]
  ∇g₁ = [1, 1] (active)

Complementary slackness:
  λ₁ · (x+y-1) = 0 ✓ (g₁ active, λ₁ > 0)
  λ₂ · (-x) = 0 ✓ (g₂ inactive, λ₂ = 0)
  λ₃ · (-y) = 0 ✓ (g₃ inactive, λ₃ = 0)

Result: Constrained minimum at
boundary of feasible region!
"""

axes[1].text(0.05, 0.5, kkt_analysis, fontsize=9, family='monospace',
             verticalalignment='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

print(kkt_analysis)

## 7. Real-World Application: Portfolio Optimization

**Problem:** Allocate investment across assets to minimize risk for a given return.

**Markowitz Portfolio Theory:**
$$\begin{align}
\min_w \quad & w^T \Sigma w \quad \text{(variance/risk)} \\
\text{s.t.} \quad & w^T \mu = r_{\text{target}} \quad \text{(expected return)} \\
& \sum_i w_i = 1 \quad \text{(fully invested)} \\
& w_i \geq 0 \quad \text{(no shorting)}
\end{align}$$

Uses **quadratic programming** (calculus-based optimization!)

In [ ]:
# Portfolio optimization example
np.random.seed(42)

# 5 assets with different returns and risks
n_assets = 5
asset_names = ['Tech', 'Finance', 'Energy', 'Healthcare', 'Consumer']

# Expected returns (annual)
mu = np.array([0.12, 0.08, 0.10, 0.09, 0.07])

# Covariance matrix (simulated)
# Generate a random positive definite covariance matrix
A = np.random.randn(n_assets, n_assets) * 0.05
Sigma = A @ A.T + np.eye(n_assets) * 0.01

def portfolio_variance(w, Sigma):
    """Portfolio variance: w^T Σ w."""
    return w @ Sigma @ w

def portfolio_return(w, mu):
    """Portfolio expected return: w^T μ."""
    return w @ mu

# Efficient frontier: vary target return
target_returns = np.linspace(mu.min(), mu.max(), 50)
efficient_portfolios = []
efficient_risks = []
efficient_weights = []

for r_target in target_returns:
    # Minimize variance subject to constraints
    constraints = [
        {'type': 'eq', 'fun': lambda w: portfolio_return(w, mu) - r_target},  # Target return
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}  # Fully invested
    ]
    bounds = [(0, 1) for _ in range(n_assets)]  # No shorting
    
    w0 = np.ones(n_assets) / n_assets  # Equal weights initial guess
    result = minimize(lambda w: portfolio_variance(w, Sigma), w0, 
                     method='SLSQP', bounds=bounds, constraints=constraints)
    
    if result.success:
        efficient_portfolios.append(result.fun)
        efficient_risks.append(np.sqrt(result.fun))  # Standard deviation
        efficient_weights.append(result.x)

efficient_risks = np.array(efficient_risks)
efficient_weights = np.array(efficient_weights)

# Find minimum variance portfolio
min_var_idx = np.argmin(efficient_risks)
min_var_return = target_returns[min_var_idx]
min_var_risk = efficient_risks[min_var_idx]

# Random portfolios for comparison
n_random = 5000
random_returns = []
random_risks = []

for _ in range(n_random):
    w = np.random.random(n_assets)
    w /= w.sum()  # Normalize
    random_returns.append(portfolio_return(w, mu))
    random_risks.append(np.sqrt(portfolio_variance(w, Sigma)))

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Efficient frontier
axes[0, 0].scatter(random_risks, random_returns, c='gray', alpha=0.3, s=5, label='Random portfolios')
axes[0, 0].plot(efficient_risks, target_returns, 'r-', linewidth=3, label='Efficient frontier')
axes[0, 0].plot(efficient_risks[min_var_idx], target_returns[min_var_idx], 'g*', 
                markersize=20, label='Min variance portfolio')

# Individual assets
for i, name in enumerate(asset_names):
    asset_risk = np.sqrt(Sigma[i, i])
    axes[0, 0].plot(asset_risk, mu[i], 'bo', markersize=8)
    axes[0, 0].text(asset_risk, mu[i], f'  {name}', fontsize=8)

axes[0, 0].set_xlabel('Risk (Standard Deviation)')
axes[0, 0].set_ylabel('Expected Return')
axes[0, 0].set_title('Efficient Frontier: Portfolio Optimization')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Asset allocation for different risk levels
indices_to_show = [0, len(efficient_weights)//4, len(efficient_weights)//2, 
                   3*len(efficient_weights)//4, -1]
x_pos = np.arange(n_assets)
width = 0.15

for i, idx in enumerate(indices_to_show):
    axes[0, 1].bar(x_pos + i*width, efficient_weights[idx], width, 
                   label=f'Return={target_returns[idx]:.1%}', alpha=0.8)

axes[0, 1].set_xlabel('Asset')
axes[0, 1].set_ylabel('Portfolio Weight')
axes[0, 1].set_title('Asset Allocation Along Efficient Frontier')
axes[0, 1].set_xticks(x_pos + 2*width)
axes[0, 1].set_xticklabels(asset_names, rotation=45)
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Covariance matrix heatmap
im = axes[1, 0].imshow(Sigma, cmap='RdYlGn_r', aspect='auto')
axes[1, 0].set_xticks(range(n_assets))
axes[1, 0].set_yticks(range(n_assets))
axes[1, 0].set_xticklabels(asset_names, rotation=45)
axes[1, 0].set_yticklabels(asset_names)
axes[1, 0].set_title('Covariance Matrix')
plt.colorbar(im, ax=axes[1, 0])

# Summary statistics
axes[1, 1].axis('off')
summary = f"""Portfolio Optimization Summary:

Assets: {', '.join(asset_names)}

Expected Returns:
"""
for name, ret in zip(asset_names, mu):
    summary += f"  {name}: {ret:.1%}\n"

summary += f"""\nMinimum Variance Portfolio:
  Return: {min_var_return:.2%}
  Risk: {min_var_risk:.2%}
  Weights:
"""
for name, w in zip(asset_names, efficient_weights[min_var_idx]):
    summary += f"    {name}: {w:.1%}\n"

summary += f"""\nKey Insights:
  • Diversification reduces risk
  • Efficient frontier shows
    best risk-return tradeoffs
  • Uses quadratic programming
  • Calculus: ∇(w^TΣw) = 2Σw
"""

axes[1, 1].text(0.1, 0.5, summary, fontsize=9, family='monospace',
                verticalalignment='center', transform=axes[1, 1].transAxes)

plt.tight_layout()
plt.show()

print(summary)

## 8. Practice Problems

### Problem 1: Constrained Optimization with Lagrange Multipliers

Minimize $f(x, y) = x^2 + 2y^2$ subject to $x + y = 10$.

Solve both analytically (using Lagrange multipliers) and numerically (using scipy).

In [ ]:
# Analytical solution:
# ℒ = x² + 2y² - λ(x + y - 10)
# ∂ℒ/∂x = 2x - λ = 0 => x = λ/2
# ∂ℒ/∂y = 4y - λ = 0 => y = λ/4
# x + y = 10 => λ/2 + λ/4 = 10 => 3λ/4 = 10 => λ = 40/3
# x = 20/3, y = 10/3

x_analytical = np.array([20/3, 10/3])
f_analytical = (20/3)**2 + 2*(10/3)**2

# Numerical solution
def f_prob1(X):
    x, y = X
    return x**2 + 2*y**2

constraint = {'type': 'eq', 'fun': lambda X: X[0] + X[1] - 10}
result = minimize(f_prob1, [5, 5], method='SLSQP', constraints=constraint)

print("Problem 1 Solution:")
print(f"\nAnalytical: x = {x_analytical[0]:.6f}, y = {x_analytical[1]:.6f}")
print(f"f(x, y) = {f_analytical:.6f}")
print(f"\nNumerical: x = {result.x[0]:.6f}, y = {result.x[1]:.6f}")
print(f"f(x, y) = {result.fun:.6f}")
print(f"\nLagrange multiplier λ = {(2*result.x[0]):.6f}")

### Problem 2: Gradient Descent Implementation

Implement gradient descent to minimize $f(x) = x^4 - 3x^3 + 2$ starting from $x_0 = 2$.

In [ ]:
def f_prob2(x):
    return x**4 - 3*x**3 + 2

def grad_prob2(x):
    return 4*x**3 - 9*x**2

# Gradient descent
x = 2.0
alpha = 0.01
history = [x]

for _ in range(1000):
    x = x - alpha * grad_prob2(x)
    history.append(x)

# Plot
x_range = np.linspace(-1, 3, 200)
y_range = f_prob2(x_range)

plt.figure(figsize=(10, 6))
plt.plot(x_range, y_range, 'b-', linewidth=2, label='f(x) = x⁴ - 3x³ + 2')
plt.plot(history, [f_prob2(x) for x in history], 'ro-', markersize=3, alpha=0.5, label='GD path')
plt.plot(history[-1], f_prob2(history[-1]), 'g*', markersize=20, label=f'Final: x={history[-1]:.4f}')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('Gradient Descent on f(x) = x⁴ - 3x³ + 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final x: {history[-1]:.6f}")
print(f"Final f(x): {f_prob2(history[-1]):.6f}")
print(f"Gradient at final: {grad_prob2(history[-1]):.6f}")

## Summary

**Key Takeaways:**

1. **Linear Programming** optimizes linear functions with linear constraints
   - Optimal solution at corner points
   - Simplex method (not calculus, but important!)

2. **Nonlinear Programming** uses **calculus** extensively
   - First-order: $\nabla f(x^*) = 0$ (necessary condition)
   - Second-order: $\nabla^2 f(x^*) \succ 0$ (sufficient condition)

3. **Lagrange Multipliers** solve constrained optimization
   - Key insight: $\nabla f = \lambda \nabla g$ at optimum
   - Geometric: gradients are parallel at boundary

4. **Gradient Descent** uses $\nabla f$ iteratively
   - Update: $x_{k+1} = x_k - \alpha \nabla f(x_k)$
   - Trade-off: learning rate affects speed vs stability

5. **Newton's Method** uses Hessian for faster convergence
   - Update: $x_{k+1} = x_k - H^{-1} \nabla f(x_k)$
   - Quadratic convergence, but expensive

6. **KKT Conditions** generalize Lagrange to inequalities
   - Stationarity, feasibility, complementary slackness
   - Foundation of modern optimization solvers

**Connection to Calculus:**
Operations Research is essentially **applied calculus** for optimization! Every method relies on derivatives (gradients, Hessians) to find optimal solutions.